# 6 — Application Out-of-Time (OOT) commune aux 3 segments (`SA`, `SB`, `SM`)

Consolidation du système de notation complet sur la base Out-of-Time :
1. Chargement unique de la base OOT, application du périmètre commun et découpage dans les 3 segments (`SA_mono_sans_eng`, `SB_mono_avec_eng`, `SM_multi`).
2. Application autonome de la chaîne de chaque segment (**SA** Cheikna, **SB** Maxime, **SM** Anis) et rattachement à l'**échelle maître unifiée** (grades `1` à `11`).
3. Pouvoir discriminant par segment et sur le **portefeuille global** (AUC-ROC, Gini, AUC-PR brute & normalisée, lift PR, KS) + perte d'information liée à l'échelle maître.
4. Contrôle de l'échelle maître sur l'OOT : monotonie, hétérogénéité inter-grades, discrimination résiduelle intra-grade.
5. **Homogénéité inter-segments** à grade égal (`SA` vs `SB` vs `SM`).
6. Backtesting de la **calibration portefeuille** (observés vs attendus, Brier & Murphy, diagramme de calibration) et export consolidé.

## 1. Chargement et segmentation unique de la base OOT

In [ ]:
# Cellule 1 — paramètres, chargement unique et segmentation OOT (SA / SB / SM)
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from utils import consolidation as cons
from utils import exploration

pl.Config.set_tbl_rows(-1); pl.Config.set_tbl_cols(-1); pl.Config.set_tbl_width_chars(200)
pl.Config.set_tbl_hide_column_data_types(True)

ROOT = Path.cwd()
ID, TARGET, DATE = "gdt", "ddefaut_ndb", "datdelhis"
SOURCE_OOT = next(
    (p for p in [ROOT / "data/_01_99__ensai_par_oot_final.sas7bdat",
                 ROOT / "data/_01_99__ensai_par_oot_final.sas7bdat.gz"] if p.exists()),
    ROOT / "data/_01_99__ensai_par_oot_final.sas7bdat",
)
CACHE_OOT = ROOT / "data/processed/base_oot_brute.parquet"

oot_cand, suivi = cons.charger_et_segmenter_oot(SOURCE_OOT, CACHE_OOT, target=TARGET, date_col=DATE, root=ROOT)
print("=== Suivi du périmètre OOT"); print(suivi["perimetre"])
print("=== Répartition par segment OOT"); print(suivi["segments"])

## 2. Scoring des 3 segments et consolidation portefeuille

In [ ]:
# Cellule 2 — application des 3 chaînes (SA, SB, SM) et contrôles d'intégrité
oot_sa = cons.scorer_segment_sa(oot_cand, root=ROOT, id_col=ID, date_col=DATE, target=TARGET)
oot_sb = cons.scorer_segment_sb(oot_cand, root=ROOT, id_col=ID, date_col=DATE, target=TARGET)
oot_sm = cons.scorer_segment_sm(oot_cand, root=ROOT, id_col=ID, date_col=DATE, target=TARGET)

oot_all = pl.concat([oot_sa, oot_sb, oot_sm], how="vertical")

# Contrôles bloquants : couverture 100 % du périmètre, aucun doublon, aucune valeur manquante
assert oot_all.height == oot_cand.height, f"Écart d'effectif : {oot_all.height} vs {oot_cand.height}"
assert oot_all.select(ID, DATE).n_unique() == oot_all.height, "Doublons (client, date) détectés"
nulls = oot_all.select("score", "pd_score", "chr", "pd_estimee", "pd_finale", "grade").null_count()
print(f"Consolidation OK : {oot_all.height:,} lignes (SA={oot_sa.height:,} | SB={oot_sb.height:,} | SM={oot_sm.height:,})")
print("Taux de défaut portefeuille OOT :", f"{oot_all[TARGET].mean():.3%}")
print("Valeurs manquantes :"); print(nulls)

## 3. Discrimination par segment et portefeuille global (AUC-ROC, Gini, AUC-PR)

In [ ]:
# Cellule 3 — discrimination OOT par segment et portefeuille + cascade échelle maître + courbes ROC / PR
disc_port = cons.synthese_discrimination_portefeuille(oot_all, target=TARGET)
print("=== Pouvoir discriminant par segment et portefeuille (OOT)")
print(disc_port)

cascade_em = cons.cascade_echelle_maitre(oot_all, target=TARGET)
print("=== Perte d'information du portefeuille (Score continu -> CHR -> Échelle maître)")
print(cascade_em)

cons.plot_roc_pr_portefeuille(oot_all, target=TARGET, col="pd_score"); plt.show()
cons.plot_roc_pr_portefeuille(oot_all, target=TARGET, col="grade"); plt.show()

## 4. Échelle maître unifiée sur l'OOT : monotonie, hétérogénéité et discrimination intra-grade

In [ ]:
# Cellule 4 — table par grade de l'échelle maître, hétérogénéité inter-grades et AUC intra-grade
table_em = cons.table_echelle_maitre_oot(oot_all, target=TARGET)
print("=== Échelle maître consolidée — OOT")
print(table_em)

hetero_em = cons.heterogeneite_grades(table_em)
print("=== Hétérogénéité entre grades adjacents — OOT")
print(hetero_em)

intra_em = cons.discrimination_intra_grade(oot_all, target=TARGET, score_col="pd_score")
print("=== Discrimination résiduelle intra-grade (pd_score) — OOT")
print(intra_em)

## 5. Homogénéité inter-segments à grade égal (`SA` vs `SB` vs `SM`)

In [ ]:
# Cellule 5 — comparaison SA / SB / SM au sein de chaque grade de l'échelle maître (7.5)
detail_inter, tests_inter = cons.homogeneite_inter_segments(oot_all, target=TARGET, n_min=30)
print("=== Tableau croisé grade x segment (taux de défaut observés OOT)")
print(detail_inter.pivot(on="segment", index="grade", values="taux_defaut").sort("grade"))
print("=== Effectifs grade x segment (OOT)")
print(detail_inter.pivot(on="segment", index="grade", values="n").sort("grade"))
print("=== Tests d'homogénéité inter-segments par grade (n >= 30)")
print(tests_inter)

fig, ax = plt.subplots(figsize=(9, 4.5))
for seg in sorted(oot_all["segment"].unique().to_list()):
    sub = detail_inter.filter((pl.col("segment") == seg) & (pl.col("n") >= 30))
    ax.plot(sub["grade"], sub["taux_defaut"] * 100, marker="o", linewidth=1.8, label=seg)
ax.plot(table_em["grade"], table_em["taux_observe"] * 100, color="black", linestyle="--", marker="s", label="PORTEFEUILLE")
ax.set(yscale="log", xlabel="grade échelle maître", ylabel="taux de défaut observé (%, log)",
       title="Homogénéité inter-segments par grade — OOT")
ax.legend(); ax.grid(alpha=0.3); plt.show()

## 6. Calibration portefeuille sur l'OOT (9.2)

In [ ]:
# Cellule 6 — observés vs attendus, Brier & Murphy par segment et global + diagramme de calibration
cal_port = cons.calibration_portefeuille(oot_all, target=TARGET)
print("=== Synthèse de calibration OOT (par segment et portefeuille)")
print(cal_port)

fig, ax = plt.subplots(figsize=(6.2, 6.2))
p_obs = table_em["taux_observe"].to_numpy()
yerr = [np.maximum(p_obs - table_em["ic_bas"].to_numpy(), 0), np.maximum(table_em["ic_haut"].to_numpy() - p_obs, 0)]
for col, lab in (("pd_estimee_moy", "PD estimée / LRA"), ("pd_finale_moy", "PD réglementaire (avec MoC)")):
    x = table_em[col].to_numpy()
    ax.errorbar(x, p_obs, yerr=yerr, fmt="o", capsize=3, label=lab)
lo = max(1e-4, min(p_obs[p_obs > 0].min(), table_em["pd_estimee_moy"].min()) / 2)
hi = min(1.0, max(p_obs.max(), table_em["pd_finale_moy"].max()) * 2)
ax.plot([lo, hi], [lo, hi], linestyle="--", color="grey", label="calibrage parfait")
ax.set(xscale="log", yscale="log", xlabel="PD calibrée moyenne", ylabel="taux de défaut observé (IC Wilson 95 %)",
       title="Diagramme de calibration portefeuille (échelle maître) — OOT")
ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.show()

## 7. Exports consolidés

In [ ]:
# Cellule 7 — export parquet de la base OOT consolidée et classeur Excel portefeuille
Path("outputs/oot").mkdir(parents=True, exist_ok=True)
oot_all.write_parquet("data/processed/base_oot_consolidee.parquet")

exploration.exporter_excel({
    "Perimetre_OOT": suivi["perimetre"],
    "Segments_OOT": suivi["segments"],
    "Discrimination": disc_port,
    "Cascade_Echelle_Maitre": cascade_em,
    "Echelle_Maitre_OOT": table_em,
    "Heterogeneite_Grades": hetero_em,
    "Intra_Grade_OOT": intra_em,
    "Inter_Segments_Detail": detail_inter,
    "Inter_Segments_Tests": tests_inter,
    "Calibration_Portefeuille": cal_port,
}, "outputs/oot/consolidation_oot_portefeuille.xlsx")
print("Exports générés : data/processed/base_oot_consolidee.parquet & outputs/oot/consolidation_oot_portefeuille.xlsx")